# JSON APIs for Digital Discourse — 2026

**DE0823 · Technical Week 4 · Google Colab · 90 minutes**

**Research question:** How can we retrieve structured online records, extract their text, and create a small reproducible discourse corpus?

**Previously:** Week 3 counted word forms in English and Latvian texts. **Today:** We acquire and inspect structured records, then reuse those counting techniques.

**Learning outcomes:** distinguish JSON text from Python objects; navigate nested records; serialize UTF-8 JSON; retrieve and validate API responses; extract text with provenance; calculate and export word-form frequencies; explain limitations.

No keys, Drive mounts, or paid services. The notebook can run offline using an embedded, clearly labeled **synthetic teaching fixture**. Run cells top to bottom.

## 1 · JSON objects and Python dictionaries (10 minutes)

JSON is a text interchange format. Python `dict` and `list` are in-memory structures. They look similar but are not identical: JSON uses `true`, `false`, `null`; Python uses `True`, `False`, `None`. JSON objects have string keys. JSON object order must not be treated as semantic information.

In [ ]:
import json
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import re
import unicodedata

record = {'id': 101, 'language': 'lv', 'title': 'Pilsētas kultūra', 'tags': ['Rīga', 'kultūra'], 'published': None, 'verified': True}
print(type(record).__name__, record['tags'][0])
json_text = json.dumps(record, ensure_ascii=False)
print(type(json_text).__name__, json_text)
assert json.loads(json_text) == record

### Nested records and missing values

Inspect keys before indexing. `dict.get()` avoids a `KeyError` for absent keys, but a missing field is not the same as an empty string or a known zero.

In [ ]:
example = {'results': [{'id': 'a1', 'metadata': {'language': 'lv'}, 'text': 'Vārdi veido diskursu.'}]}
print(example.keys())
print(example['results'][0]['metadata'].get('language'))
print(example['results'][0].get('author'))  # unknown, not an empty author
try:
    print(example['results'][0]['missing'])
except KeyError as exc:
    print('Expected missing-key example:', exc)

## 2 · Serialization and Unicode (15 minutes)

`json.dumps` produces JSON text; `json.loads` parses JSON text. `json.dump` writes to a stream; `json.load` reads from a stream. `ensure_ascii=False` preserves visible Latvian letters; JSON with escaped Unicode is also valid. Always specify UTF-8 for disk files.

In [ ]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
example_path = output_dir / 'example_record.json'
with example_path.open('w', encoding='utf-8') as f:
    json.dump(record, f, ensure_ascii=False, indent=2)
with example_path.open(encoding='utf-8') as f:
    restored = json.load(f)
assert restored == record
print(example_path.read_text(encoding='utf-8'))

### Quick check (think–pair–share)

1. What Python type results from `json.loads('[1, 2, 3]')`?
2. Why is `json.dumps({'ok': True})` different from `str({'ok': True})`?
3. What does `ensure_ascii=False` change, and what does it *not* change?

In [ ]:
assert isinstance(json.loads('[1, 2, 3]'), list)
assert json.dumps({'ok': True}) == '{"ok": true}'
print('Checks passed')

## 3 · HTTP APIs and defensive retrieval (20 minutes)

An API is an interface to a service. A GET request asks for a resource. HTTP 200 means success at the HTTP level, **not** that the data has the expected structure. Use a timeout, check status, decode JSON, and validate the resulting shape.

The first example uses [JSONPlaceholder](https://jsonplaceholder.typicode.com/), a fake REST API for practice. Its records are **not real discourse evidence**.

In [ ]:
import requests

def fetch_json(url, *, params=None, timeout=12):
    '''Return parsed JSON; raise on network, HTTP, or JSON errors.'''
    response = requests.get(url, params=params, timeout=timeout, headers={'Accept': 'application/json', 'User-Agent': 'DE0823-teaching-notebook/2026'})
    response.raise_for_status()
    return response.json()

print('requests version:', requests.__version__)

In [ ]:
try:
    demo_users = fetch_json('https://jsonplaceholder.typicode.com/users')
    assert isinstance(demo_users, list) and demo_users
    print('Users:', len(demo_users), '| first:', demo_users[0].get('name'))
    print('Nested city:', demo_users[0].get('address', {}).get('city'))
except (requests.RequestException, ValueError, AssertionError) as exc:
    demo_users = []
    print('Live introductory API unavailable; continue with the local corpus:', type(exc).__name__)

## 4 · Digital discourse case study (20 minutes)

Use the **MediaWiki REST API** for short article summaries in two languages. This is encyclopedic discourse, **not** social-media discourse. We request a small, explicitly chosen set of articles; it is a convenience sample, not a representative corpus.

**Reliability rule:** The notebook has an embedded synthetic fallback with explicit provenance. If the network or endpoint fails, the analysis still runs, but the outputs must not be presented as retrieved Wikipedia content. API payload fields may change; validate before using them.

In [ ]:
PAGES = [('en', 'Digital_humanities'), ('en', 'Discourse_analysis'), ('lv', 'Rīga'), ('lv', 'Latviešu_valoda')]
# Clearly synthetic offline fixture, not quotations or actual Wikipedia extracts.
OFFLINE_FIXTURE = [
    {'id':'fixture-en-1','language':'en','title':'Digital humanities','text':'Digital humanities studies digital sources and research methods. Researchers examine texts and metadata.'},
    {'id':'fixture-en-2','language':'en','title':'Discourse analysis','text':'Discourse analysis examines language and context. Texts can be compared using careful methods.'},
    {'id':'fixture-lv-1','language':'lv','title':'Rīga','text':'Rīga ir Latvijas pilsēta. Pilsētas vēsture un kultūra ir pētniecības tēmas.'},
    {'id':'fixture-lv-2','language':'lv','title':'Latviešu valoda','text':'Latviešu valoda ir pētniecības objekts. Valoda un teksts palīdz analizēt diskursu.'},
]

def extract_summary(payload, language, slug):
    if not isinstance(payload, dict):
        raise ValueError('Expected JSON object')
    text = payload.get('extract')
    title = payload.get('title')
    if not isinstance(text, str) or not text.strip() or not isinstance(title, str):
        raise ValueError('Unexpected summary schema')
    return {'id': f'{language}:{slug}', 'language': language, 'title': title, 'text': text.strip()}

assert len(OFFLINE_FIXTURE) == 4

In [ ]:
from urllib.parse import quote

USE_LIVE_API = True  # set False for guaranteed offline demonstration
retrieved_at = datetime.now(timezone.utc).isoformat()
records = []
source_urls = []
source_kind = 'synthetic_fixture'
if USE_LIVE_API:
    try:
        for lang, slug in PAGES:
            url = f'https://{lang}.wikipedia.org/api/rest_v1/page/summary/{quote(slug)}'
            payload = fetch_json(url)
            records.append(extract_summary(payload, lang, slug))
            source_urls.append(url)
        source_kind = 'mediawiki_live'
    except (requests.RequestException, ValueError, KeyError) as exc:
        print('Live API failed; using synthetic teaching fixture:', type(exc).__name__)
        records = []
        source_urls = []
if not records:
    records = [dict(item) for item in OFFLINE_FIXTURE]
    source_urls = ['embedded:OFFLINE_FIXTURE']
print('Source:', source_kind, '| records:', len(records))
for item in records:
    print(item['language'], item['title'], '—', item['text'][:90])

### Provenance and limitations

Record retrieval time and exact source endpoints. The retrieval time is *not* the article publication date. API data can change. A fallback fixture is pedagogical test data and must not be cited as real-world evidence. Language-specific word-form counts are not directly comparable without considering morphology and corpus size.

In [ ]:
corpus = {'metadata': {'course': 'DE0823', 'retrieved_at_utc': retrieved_at, 'source_kind': source_kind, 'source_urls': source_urls, 'selection': 'four preselected page titles; convenience sample', 'extraction': 'MediaWiki summary extract field or synthetic teaching text', 'limitations': 'Not representative; summary text only; synthetic fallback is not empirical evidence'}, 'records': records}
assert all(isinstance(r.get('text'), str) and r['text'].strip() and r.get('language') in ('en','lv') for r in corpus['records'])
corpus_path = output_dir / 'discourse_corpus.json'
with corpus_path.open('w', encoding='utf-8') as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)
print('Saved:', corpus_path)

## 5 · From JSON to word frequencies (15 minutes)

Reuse the Week 3 idea: Unicode normalization, case folding, tokenization, and `Counter`. This simple tokenizer counts surface word forms, not lemmas. For Latvian, inflection creates many forms of the same lexeme. Counts across languages should be interpreted separately.

In [ ]:
def tokenize(text):
    normalized = unicodedata.normalize('NFC', text).casefold()
    # Extract Unicode letter sequences, avoiding digits and underscores.
    return re.findall(r'[^\W\d_]+(?:[\'’\-][^\W\d_]+)*', normalized, flags=re.UNICODE)

frequencies = {}
for lang in sorted({r['language'] for r in records}):
    tokens = [token for r in records if r['language'] == lang for token in tokenize(r['text'])]
    frequencies[lang] = Counter(tokens)
    print(lang, '| tokens:', len(tokens), '| distinct word forms:', len(frequencies[lang]), '| top:', frequencies[lang].most_common(8))
assert all(sum(c.values()) > 0 for c in frequencies.values())

In [ ]:
import csv
frequency_path = output_dir / 'word_frequencies.tsv'
with frequency_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['language', 'word_form', 'count'])
    for lang, counts in sorted(frequencies.items()):
        for word, count in sorted(counts.items(), key=lambda pair: (-pair[1], pair[0])):
            writer.writerow([lang, word, count])
with frequency_path.open(encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f, delimiter='\t'))
assert rows and set(rows[0]) == {'language','word_form','count'}
assert sum(int(r['count']) for r in rows) == sum(sum(c.values()) for c in frequencies.values())
print('Saved:', frequency_path, '| rows:', len(rows))

## 6 · Student exercises and interpretation (10 minutes)

**Exercise A (5 minutes):** Change `USE_LIVE_API` to `False`, run from the retrieval cell onward, and inspect the provenance fields in `outputs/discourse_corpus.json`. Explain why the source label changes.

**Exercise B (5 minutes):** Identify the three most frequent word forms in each language. Explain why these results cannot establish which topics dominate English-language versus Latvian-language discourse.

**Optional extension:** Add one article title to `PAGES` and rerun. What happens if the title does not exist? How would you handle partial success without mixing datasets silently?

**Reflection:** What is lost when a document is reduced to its summary text and then to a bag of word forms? Consider genre, authorship, context, selection bias, tokenization, and source revisions.

In [ ]:
print('Final validation')
with corpus_path.open(encoding='utf-8') as f:
    saved = json.load(f)
assert saved['metadata']['source_kind'] in {'mediawiki_live','synthetic_fixture'}
assert len(saved['records']) == len(records)
assert all(r['id'] and r['title'] for r in saved['records'])
print('OK: reproducible corpus JSON and TSV created in outputs/')

## Further reading and instructor notes

- [Python `json` documentation](https://docs.python.org/3/library/json.html)
- [Requests quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [MediaWiki API documentation](https://www.mediawiki.org/wiki/API:Main_page)
- [Week 3: CountingWords_2026](../TextProcessing/CountingWords_2026.ipynb)

**Instructor:** Run all once with `USE_LIVE_API=False` before class, then try live mode. Demonstrate `type()`, nested access, and `.get()` slowly. Ask students to distinguish HTTP success from schema validity, and real data from synthetic fixtures. Do not interpret the toy sample as empirical discourse research. Outputs live in the temporary Colab filesystem; download them if needed. Next session: web scraping and pandas.